In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import datetime as dt
import boto3

try:
    import snowflake.connector
except:
    ! pip install snowflake-connector-python
    import snowflake.connector

from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, dsa
from cryptography.hazmat.primitives import serialization

In [ ]:
str_dtm_today = str(dt.datetime.now())
print(f'Latest run date: {str_dtm_today}')

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dtm_min = 'YYYY-MM-DD'

#### Connect to snowflake

In [ ]:
# load 
str_filename = 'rsa_key.p8'
str_local_path = f'./{str_filename}'
with open(str_local_path, "rb") as key:
    p_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend(),
    )
# convert to bytes
private_key = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
# connect to snowflake
conn = snowflake.connector.connect(
    user='username', 
    private_key=private_key,
)

#### Query

In [ ]:
str_query = f"""
SELECT *
FROM
TABLE
"""

#### Pull data

In [ ]:
%%time

# pull payloads
df = pd.read_sql(
    sql=str_query,
    con=conn,
)
# show
df

#### Close connection

In [ ]:
conn.close()

#### Write to s3

In [ ]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)